# F1 Undercut Strategy Prediction
---
**Research Question:** Given a driver's current gap to the car ahead and their respective tire ages, how accurately can I predict whether pitting early, the undercut, will lead to a net position gain over that car?

**Approach:**
- **Label:** Undercut success = driver beats the *specific car that was directly ahead* after both cars have cycled through their stops, with position resolved 4 laps after the car-ahead pits.
- **Models:** Logistic Regression as the interpretable baseline, plus XGBoost as the main model.
- **Holdout:** Train on 2022–2023, evaluate on 2024 with a temporal split.
- **Features:** Gap ahead, tire ages, pace delta, degradation rates, closing rate, pit loss, race progress

---


In [ ]:
import fastf1
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.pipeline import Pipeline

import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for script execution
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Resolve project root (works whether notebook runs from notebooks/ or root) ──
import os
_nb_dir = os.path.abspath('')
ROOT = os.path.dirname(_nb_dir) if os.path.basename(_nb_dir) == 'notebooks' else _nb_dir
CACHE_DIR   = os.path.join(ROOT, 'f1-cache')
DATA_DIR    = os.path.join(ROOT, 'data')
MODEL_DIR   = os.path.join(ROOT, 'models')
FIGURES_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

fastf1.Cache.enable_cache(CACHE_DIR)

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-darkgrid')

print('FastF1:', fastf1.__version__)
print('XGBoost:', xgb.__version__)

In [ ]:
# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────

FUEL_BURN_RATE   = 1.8    # kg per lap
FUEL_LAP_EFFECT  = 0.035  # seconds per kg

PACE_WINDOW      = 4      # laps for rolling pace average
DEG_WINDOW       = 5      # laps for degradation slope
STABILIZATION    = 4      # laps after car-ahead pits to resolve outcome
GAP_WINDOW       = 3      # laps for closing rate trend
MAX_GAP          = 30.0   # s — ignore gaps > 30s (not a real race battle)
MIN_HIST_LAPS    = 3      # minimum laps of history before pit to compute features

TRAIN_YEARS      = [2022, 2023]
TEST_YEARS       = [2024]
ALL_YEARS        = TRAIN_YEARS + TEST_YEARS

DATASET_PATH     = os.path.join(DATA_DIR,  'f1_undercut_dataset.csv')
MODEL_PATH       = os.path.join(MODEL_DIR, 'f1_undercut_model.pkl')

In [ ]:
# ─────────────────────────────────────────────
# Helper: fuel correction
# ─────────────────────────────────────────────

def fuel_correct(lap_seconds, lap_number):
    """Remove fuel weight effect to isolate tyre degradation."""
    return lap_seconds - (lap_number - 1) * FUEL_BURN_RATE * FUEL_LAP_EFFECT


# ─────────────────────────────────────────────
# Helper: gap timeseries (vectorised)
# ─────────────────────────────────────────────

def build_gap_timeseries(session):
    """
    For every (driver, lap) return gap_ahead, gap_behind, car_ahead_driver.

    Uses LapStartTime differences: the driver who crossed the S/F line
    earlier is further ahead. Only accurate laps are included.
    """
    laps = session.laps.copy()
    laps = laps[laps['IsAccurate'] == True].copy()
    laps = laps.dropna(subset=['LapStartTime'])

    # Sort: within each lap number, earliest LapStartTime = race leader
    laps = laps.sort_values(['LapNumber', 'LapStartTime']).reset_index(drop=True)

    # Vectorised shift within each lap group
    grp = laps.groupby('LapNumber', sort=False)

    laps['ahead_LapStartTime'] = grp['LapStartTime'].shift(1)
    laps['car_ahead_driver']   = grp['Driver'].shift(1)
    laps['behind_LapStartTime'] = grp['LapStartTime'].shift(-1)

    laps['gap_ahead'] = (
        laps['LapStartTime'] - laps['ahead_LapStartTime']
    ).dt.total_seconds()

    laps['gap_behind'] = (
        laps['behind_LapStartTime'] - laps['LapStartTime']
    ).dt.total_seconds()

    return laps[['Driver', 'LapNumber', 'gap_ahead', 'gap_behind',
                 'car_ahead_driver']].copy()


# ─────────────────────────────────────────────
# Helper: pit lane time loss
# ─────────────────────────────────────────────

def compute_pit_loss(session):
    """
    Median time from PitInTime to PitOutTime across all stops.
    PitInTime is on the in-lap, PitOutTime is on the out-lap (next row).
    """
    laps = session.laps.copy()
    in_laps  = laps[laps['PitInTime'].notna()][['Driver', 'LapNumber', 'PitInTime']].copy()
    out_laps = laps[laps['PitOutTime'].notna()][['Driver', 'LapNumber', 'PitOutTime']].copy()

    in_laps['OutLapNumber'] = in_laps['LapNumber'] + 1
    out_laps = out_laps.rename(columns={'LapNumber': 'OutLapNumber'})

    merged = in_laps.merge(out_laps, on=['Driver', 'OutLapNumber'], how='inner')
    if merged.empty:
        return 22.0  # circuit-generic fallback

    durations = (merged['PitOutTime'] - merged['PitInTime']).dt.total_seconds()
    return float(durations.median())


# ─────────────────────────────────────────────
# Helper: rolling pace
# ─────────────────────────────────────────────

def get_pace(driver_laps, up_to_lap, window=PACE_WINDOW):
    """Median fuel-corrected lap time over the last `window` green laps."""
    recent = driver_laps[
        (driver_laps['LapNumber'] < up_to_lap) &
        (driver_laps['LapNumber'] >= up_to_lap - window) &
        (driver_laps['IsAccurate'] == True) &
        (driver_laps['TrackStatus'].astype(str) == '1')
    ].copy()
    if recent.empty:
        return np.nan
    lt = recent['LapTime'].dt.total_seconds()
    corrected = lt - (recent['LapNumber'] - 1) * FUEL_BURN_RATE * FUEL_LAP_EFFECT
    return float(corrected.median())


# ─────────────────────────────────────────────
# Helper: tyre degradation slope
# ─────────────────────────────────────────────

def get_deg_delta(driver_laps, up_to_lap, window=DEG_WINDOW):
    """Slope (s/lap) of fuel-corrected lap time vs tyre age over last `window` green laps."""
    recent = driver_laps[
        (driver_laps['LapNumber'] < up_to_lap) &
        (driver_laps['LapNumber'] >= up_to_lap - window) &
        (driver_laps['IsAccurate'] == True) &
        (driver_laps['TrackStatus'].astype(str) == '1') &
        (driver_laps['TyreLife'].notna())
    ].copy()
    if len(recent) < 3:
        return np.nan
    lt = recent['LapTime'].dt.total_seconds()
    corrected = lt - (recent['LapNumber'] - 1) * FUEL_BURN_RATE * FUEL_LAP_EFFECT
    x = recent['TyreLife'].values.astype(float)
    y = corrected.values
    if np.ptp(x) < 0.5:   # all same tyre age — no slope
        return 0.0
    return float(np.polyfit(x, y, 1)[0])


# ─────────────────────────────────────────────
# Helper: SC / VSC contamination check
# ─────────────────────────────────────────────

def has_sc_between(all_laps, driver, lap_start, lap_end):
    """True if any Safety Car or VSC period occurs in [lap_start, lap_end]."""
    window = all_laps[
        (all_laps['Driver'] == driver) &
        (all_laps['LapNumber'] >= lap_start) &
        (all_laps['LapNumber'] <= lap_end)
    ]['TrackStatus'].dropna().astype(str)
    return any(any(c in s for c in ['4', '5', '6']) for s in window)

In [ ]:
# ─────────────────────────────────────────────
# Core: extract undercut records from one session
# ─────────────────────────────────────────────

def build_undercut_records(session, year, circuit_name):
    """
    For each pit stop in the session where the driver's car-ahead
    has not yet pitted, record features at the decision lap and
    label whether the undercut succeeded.

    Outcome resolution: 4 laps after the car-ahead completes its
    next pit stop, compare relative race positions.
    SC/VSC-contaminated windows are discarded.
    """
    laps = session.laps.copy()
    laps = laps[laps['LapTime'].notna()].copy()
    total_laps = int(laps['LapNumber'].max())

    gaps_df  = build_gap_timeseries(session)
    pit_loss = compute_pit_loss(session)

    # Index for fast lookup
    gaps_idx = gaps_df.set_index(['Driver', 'LapNumber'])
    laps_idx  = laps.set_index(['Driver', 'LapNumber'])

    records = []

    pit_rows = laps[laps['PitInTime'].notna()][['Driver', 'LapNumber']].copy()

    for _, pit_row in pit_rows.iterrows():
        driver   = pit_row['Driver']
        pit_lap  = int(pit_row['LapNumber'])
        dec_lap  = pit_lap - 1   # decision lap (just before pitting)

        if dec_lap < MIN_HIST_LAPS:
            continue

        # ── Gap info at decision lap ──────────────────────────────
        try:
            gap_row    = gaps_idx.loc[(driver, dec_lap)]
            # If multiple rows (shouldn't happen), take first
            if isinstance(gap_row, pd.DataFrame):
                gap_row = gap_row.iloc[0]
        except KeyError:
            continue

        car_ahead = gap_row['car_ahead_driver']
        gap_ahead = gap_row['gap_ahead']

        if car_ahead is None or pd.isna(car_ahead) or pd.isna(gap_ahead):
            continue   # driver is leading
        if gap_ahead > MAX_GAP or gap_ahead <= 0:
            continue   # not an active battle

        # ── Confirm car-ahead pits AFTER the driver ───────────────
        car_ahead_pits = laps[
            (laps['Driver'] == car_ahead) &
            (laps['PitInTime'].notna()) &
            (laps['LapNumber'] > pit_lap)
        ]['LapNumber']

        if car_ahead_pits.empty:
            continue   # car-ahead never pits after — not an undercut

        ca_pit_lap  = int(car_ahead_pits.min())
        eval_lap    = min(ca_pit_lap + STABILIZATION, total_laps)

        # ── SC/VSC contamination check ────────────────────────────
        if has_sc_between(laps, driver, pit_lap, eval_lap):
            continue

        # ── Outcome: relative order at eval_lap ───────────────────
        eval_slice = gaps_df[
            gaps_df['LapNumber'] == eval_lap
        ].sort_values('gap_ahead')  # sorted by LapStartTime order

        # Reconstruct order from cumulative LapStartTime gaps
        eval_laps_session = laps[
            laps['LapNumber'] == eval_lap
        ][['Driver', 'LapStartTime']].dropna().sort_values('LapStartTime').reset_index(drop=True)

        if eval_laps_session.empty:
            continue

        order = {row['Driver']: idx for idx, row in eval_laps_session.iterrows()}

        if driver not in order or car_ahead not in order:
            continue   # one retired

        # Lower index = further ahead in race
        undercut_success = 1 if order[driver] < order[car_ahead] else 0

        # ── Feature extraction ────────────────────────────────────
        driver_laps = laps[laps['Driver'] == driver].sort_values('LapNumber')
        ca_laps     = laps[laps['Driver'] == car_ahead].sort_values('LapNumber')

        own_pace        = get_pace(driver_laps, dec_lap + 1)
        threat_pace     = get_pace(ca_laps,     dec_lap + 1)
        deg_delta       = get_deg_delta(driver_laps, dec_lap + 1)
        ca_deg_delta    = get_deg_delta(ca_laps,     dec_lap + 1)

        # Tire ages at decision lap
        try:
            dec_row    = laps_idx.loc[(driver, dec_lap)]
            if isinstance(dec_row, pd.DataFrame): dec_row = dec_row.iloc[0]
            ca_dec_row = laps_idx.loc[(car_ahead, dec_lap)]
            if isinstance(ca_dec_row, pd.DataFrame): ca_dec_row = ca_dec_row.iloc[0]
        except KeyError:
            continue

        tire_age    = dec_row.get('TyreLife',  np.nan)
        compound    = dec_row.get('Compound',  'UNKNOWN')
        ca_tire_age = ca_dec_row.get('TyreLife', np.nan)

        # Closing rate: slope of gap_ahead over last GAP_WINDOW laps
        recent_gaps = gaps_df[
            (gaps_df['Driver'] == driver) &
            (gaps_df['LapNumber'] >= dec_lap - GAP_WINDOW) &
            (gaps_df['LapNumber'] <= dec_lap)
        ].sort_values('LapNumber').dropna(subset=['gap_ahead'])

        if len(recent_gaps) >= 2:
            closing_rate = float(np.polyfit(
                recent_gaps['LapNumber'].values,
                recent_gaps['gap_ahead'].values, 1
            )[0])   # negative = closing on car ahead
        else:
            closing_rate = np.nan

        pace_delta         = (own_pace - threat_pace) if not pd.isna(own_pace) and not pd.isna(threat_pace) else np.nan
        tire_age_advantage = (float(ca_tire_age) - float(tire_age)) if not pd.isna(ca_tire_age) and not pd.isna(tire_age) else np.nan
        pit_loss_fraction  = pit_loss / own_pace if not pd.isna(own_pace) and own_pace > 0 else np.nan

        records.append({
            # Metadata (not model features)
            'year':        year,
            'circuit':     circuit_name,
            'driver':      driver,
            'car_ahead':   car_ahead,
            'pit_lap':     pit_lap,
            # Features
            'gap_ahead':          gap_ahead,
            'tire_age':           float(tire_age) if not pd.isna(tire_age) else np.nan,
            'car_ahead_tire_age': float(ca_tire_age) if not pd.isna(ca_tire_age) else np.nan,
            'tire_age_advantage': tire_age_advantage,
            'compound':           str(compound),
            'own_pace':           own_pace,
            'threat_pace':        threat_pace,
            'pace_delta':         pace_delta,
            'deg_delta':          deg_delta,
            'ca_deg_delta':       ca_deg_delta,
            'closing_rate':       closing_rate,
            'pit_loss':           pit_loss,
            'pit_loss_fraction':  pit_loss_fraction,
            'race_progress':      dec_lap / total_laps if total_laps > 0 else np.nan,
            # Label
            'undercut_success':   undercut_success,
        })

    return records

In [ ]:
# ─────────────────────────────────────────────
# Dataset construction loop
# ─────────────────────────────────────────────

def build_full_dataset(years):
    all_records = []
    for year in years:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        gp_names = schedule['EventName'].tolist()
        print(f'\n── {year}: {len(gp_names)} races ──')
        for gp in gp_names:
            try:
                session = fastf1.get_session(year, gp, 'R')
                session.load(telemetry=False, weather=False, messages=False)
                records = build_undercut_records(session, year, gp)
                all_records.extend(records)
                print(f'  {gp:40s}  {len(records):3d} undercut attempts')
            except Exception as e:
                print(f'  {gp:40s}  FAILED: {e}')
    return pd.DataFrame(all_records)


if Path(DATASET_PATH).exists():
    print('Loading cached dataset...')
    df = pd.read_csv(DATASET_PATH)
else:
    print('Building dataset — this will take several minutes...')
    df = build_full_dataset(ALL_YEARS)
    df.to_csv(DATASET_PATH, index=False)
    print(f'\nSaved to {DATASET_PATH}')

print(f'\nDataset shape: {df.shape}')
print(f'Years covered: {sorted(df["year"].unique())}')
print(f'\nLabel distribution:')
vc = df['undercut_success'].value_counts()
print(f'  Success (1): {vc.get(1, 0):,}  ({vc.get(1,0)/len(df):.1%})')
print(f'  Failure (0): {vc.get(0, 0):,}  ({vc.get(0,0)/len(df):.1%})')

In [ ]:
# ─────────────────────────────────────────────
# Feature engineering & cleaning
# ─────────────────────────────────────────────

# One-hot encode compound
compound_dummies = pd.get_dummies(df['compound'], prefix='compound')
df_feat = pd.concat([df, compound_dummies], axis=1)

BASE_FEATURES = [
    'gap_ahead',
    'tire_age',
    'car_ahead_tire_age',
    'tire_age_advantage',
    'own_pace',
    'threat_pace',
    'pace_delta',
    'deg_delta',
    'ca_deg_delta',
    'closing_rate',
    'pit_loss',
    'pit_loss_fraction',
    'race_progress',
]

FEATURES = BASE_FEATURES + [c for c in compound_dummies.columns]

# Drop rows missing the most important features
REQUIRED = ['gap_ahead', 'tire_age', 'car_ahead_tire_age', 'own_pace', 'threat_pace']
df_model = df_feat.dropna(subset=REQUIRED + ['undercut_success']).copy()

# Fill remaining NaNs with column median
for col in FEATURES:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna(df_model[col].median())
    else:
        df_model[col] = 0   # compound not present in data → all zeros

print(f'Modelling dataset: {len(df_model):,} samples, {len(FEATURES)} features')
print(f'Class balance: {df_model["undercut_success"].mean():.1%} successful undercuts')
df_model.head(3)

In [ ]:
# ─────────────────────────────────────────────
# EDA — distributions
# ─────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

eda_features = [
    'gap_ahead', 'tire_age', 'car_ahead_tire_age', 'tire_age_advantage',
    'pace_delta', 'deg_delta', 'closing_rate', 'race_progress'
]
labels_map = {1: 'Success', 0: 'Failure'}
colours    = {1: '#2ecc71', 0: '#e74c3c'}

for ax, feat in zip(axes, eda_features):
    for label, name in labels_map.items():
        subset = df_model[df_model['undercut_success'] == label][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=colours[label], label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions by Undercut Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_eda.png'), dpi=150, bbox_inches='tight')
plt.show()
print('EDA saved to undercut_eda.png')

In [ ]:
# ─────────────────────────────────────────────
# EDA — gap ahead vs success rate (binned)
# ─────────────────────────────────────────────

df_model['gap_bin'] = pd.cut(df_model['gap_ahead'], bins=[0, 2, 4, 6, 8, 12, 20, 30])
gap_success = df_model.groupby('gap_bin', observed=True)['undercut_success'].agg(['mean', 'count'])
gap_success.columns = ['success_rate', 'n_attempts']

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(range(len(gap_success)), gap_success['success_rate'],
        color='steelblue', alpha=0.8, label='Success Rate')
ax2.plot(range(len(gap_success)), gap_success['n_attempts'],
         'o--', color='tomato', label='# Attempts')

ax1.set_xticks(range(len(gap_success)))
ax1.set_xticklabels([str(b) for b in gap_success.index], rotation=30)
ax1.set_xlabel('Gap to Car Ahead (s)')
ax1.set_ylabel('Undercut Success Rate', color='steelblue')
ax2.set_ylabel('Number of Attempts', color='tomato')
ax1.set_title('Undercut Success Rate by Gap to Car Ahead', fontsize=12, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_gap_vs_success.png'), dpi=150, bbox_inches='tight')
plt.show()
print(gap_success.to_string())

In [ ]:
# ─────────────────────────────────────────────
# EDA — tire age advantage vs success rate
# ─────────────────────────────────────────────

df_model['age_adv_bin'] = pd.cut(
    df_model['tire_age_advantage'],
    bins=[-30, -10, -5, 0, 5, 10, 20, 40]
)
age_success = df_model.groupby('age_adv_bin', observed=True)['undercut_success'].agg(['mean', 'count'])
age_success.columns = ['success_rate', 'n_attempts']

fig, ax = plt.subplots(figsize=(10, 5))
colours_bar = ['#c0392b' if r < 0.5 else '#27ae60' for r in age_success['success_rate']]
ax.bar(range(len(age_success)), age_success['success_rate'],
       color=colours_bar, alpha=0.85)
ax.axhline(0.5, color='black', linestyle='--', linewidth=1, label='50% baseline')
ax.set_xticks(range(len(age_success)))
ax.set_xticklabels([str(b) for b in age_success.index], rotation=30)
ax.set_xlabel('Tire Age Advantage (car_ahead_age − driver_age, laps)')
ax.set_ylabel('Undercut Success Rate')
ax.set_title('Undercut Success Rate by Relative Tire Age Advantage', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_age_vs_success.png'), dpi=150, bbox_inches='tight')
plt.show()
print(age_success.to_string())

In [ ]:
# ─────────────────────────────────────────────
# Train / test split (temporal)
# ─────────────────────────────────────────────

train_mask = df_model['year'].isin(TRAIN_YEARS)
test_mask  = df_model['year'].isin(TEST_YEARS)

X_train = df_model.loc[train_mask, FEATURES].values.astype(float)
y_train = df_model.loc[train_mask, 'undercut_success'].values.astype(int)

X_test  = df_model.loc[test_mask,  FEATURES].values.astype(float)
y_test  = df_model.loc[test_mask,  'undercut_success'].values.astype(int)

print(f'Train (2022–2023): {len(X_train):,} samples  |  '
      f'Success rate: {y_train.mean():.1%}')
print(f'Test  (2024):      {len(X_test):,} samples  |  '
      f'Success rate: {y_test.mean():.1%}')

In [ ]:
# ─────────────────────────────────────────────
# Model 1: Logistic Regression (baseline)
# ─────────────────────────────────────────────

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE))
])
lr_pipe.fit(X_train, y_train)

lr_preds = lr_pipe.predict(X_test)
lr_probs = lr_pipe.predict_proba(X_test)[:, 1]

print('=== Logistic Regression (Baseline) ===')
print(f'Accuracy:  {accuracy_score(y_test, lr_preds):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, lr_probs):.3f}')
print()
print(classification_report(y_test, lr_preds, target_names=['No Gain', 'Position Gain']))

# Coefficients
coef_df = pd.DataFrame({
    'feature':     FEATURES,
    'coefficient': lr_pipe.named_steps['clf'].coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print('\nTop 10 logistic regression coefficients:')
print(coef_df.head(10).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# Model 2: XGBoost (primary)
# ─────────────────────────────────────────────

# Balance classes
neg  = (y_train == 0).sum()
pos  = (y_train == 1).sum()
spw  = neg / pos if pos > 0 else 1.0

xgb_model = xgb.XGBClassifier(
    n_estimators      = 400,
    max_depth         = 4,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = spw,
    min_child_weight  = 5,
    random_state      = RANDOM_STATE,
    eval_metric       = 'logloss',
    verbosity         = 0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost (Primary) ===')
print(f'Accuracy:  {accuracy_score(y_test, xgb_preds):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, xgb_probs):.3f}')
print()
print(classification_report(y_test, xgb_preds, target_names=['No Gain', 'Position Gain']))

# Save model
joblib.dump({'model': xgb_model, 'features': FEATURES}, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

In [ ]:
# ─────────────────────────────────────────────
# Evaluation plots
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix — LR
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_preds,
    display_labels=['No Gain', 'Position Gain'],
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Logistic Regression\nConfusion Matrix', fontsize=11)

# Confusion matrix — XGBoost
ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_preds,
    display_labels=['No Gain', 'Position Gain'],
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('XGBoost\nConfusion Matrix', fontsize=11)

# ROC curves
RocCurveDisplay.from_predictions(y_test, lr_probs,  ax=axes[2], name='Logistic Regression')
RocCurveDisplay.from_predictions(y_test, xgb_probs, ax=axes[2], name='XGBoost')
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[2].set_title('ROC Curves — 2024 Holdout', fontsize=11)

plt.suptitle('Undercut Success Prediction — Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_model_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Evaluation saved to undercut_model_evaluation.png')

In [ ]:
# ─────────────────────────────────────────────
# SHAP feature importance
# ─────────────────────────────────────────────

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURES,
    show=False, max_display=12, plot_type='bar'
)
axes[0].set_title('Mean |SHAP| — Feature Importance', fontsize=11)

plt.sca(axes[1])
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURES,
    show=False, max_display=12
)
axes[1].set_title('SHAP Beeswarm — Direction & Magnitude', fontsize=11)

plt.suptitle('XGBoost SHAP Analysis — Drivers of Undercut Success', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_shap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('SHAP plot saved to undercut_shap.png')

In [ ]:
# ─────────────────────────────────────────────
# Probability calibration: P(success) vs actual rate
# ─────────────────────────────────────────────

prob_bins = np.linspace(0, 1, 11)
bin_indices = np.digitize(xgb_probs, prob_bins) - 1
bin_indices = np.clip(bin_indices, 0, len(prob_bins) - 2)

cal_df = pd.DataFrame({'prob': xgb_probs, 'actual': y_test, 'bin': bin_indices})
cal_stats = cal_df.groupby('bin').agg(
    mean_prob=('prob',   'mean'),
    actual_rate=('actual', 'mean'),
    n=('actual', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(cal_stats['mean_prob'], cal_stats['actual_rate'],
           s=cal_stats['n'] * 3, alpha=0.8, color='steelblue', label='Model bins')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Actual Success Rate')
ax.set_title('XGBoost Probability Calibration\n(bubble size ∝ sample count)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('undercut_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Model comparison summary
# ─────────────────────────────────────────────

from sklearn.metrics import precision_score, recall_score, f1_score

summary = pd.DataFrame([
    {
        'Model':     'Logistic Regression (baseline)',
        'Accuracy':  round(accuracy_score(y_test, lr_preds),  3),
        'ROC-AUC':   round(roc_auc_score(y_test, lr_probs),   3),
        'Precision': round(precision_score(y_test, lr_preds), 3),
        'Recall':    round(recall_score(y_test, lr_preds),    3),
        'F1':        round(f1_score(y_test, lr_preds),        3),
    },
    {
        'Model':     'XGBoost (primary)',
        'Accuracy':  round(accuracy_score(y_test, xgb_preds),  3),
        'ROC-AUC':   round(roc_auc_score(y_test, xgb_probs),   3),
        'Precision': round(precision_score(y_test, xgb_preds), 3),
        'Recall':    round(recall_score(y_test, xgb_preds),    3),
        'F1':        round(f1_score(y_test, xgb_preds),        3),
    },
])

print('=== Final Model Comparison (2024 Holdout) ===')
print(summary.to_string(index=False))

---
## Key Findings

### Predictive Accuracy
The XGBoost model predicts undercut success with meaningful accuracy on the 2024 holdout. The ROC-AUC above 0.5 tells me the model is learning real signal beyond the base rate, so strategy is not just random noise here. The gap between XGBoost and Logistic Regression also tells me how much non-linear interaction the data contains. Essentially, a large gap plus fresh tires is not the same problem as a large gap plus worn tires.

### Most Predictive Features (from SHAP)
1. **`tire_age_advantage`** , how many more laps the car ahead has on their tires compared with the pitting driver. The bigger this number is, the more the car ahead tends to fall away after the stop, which makes the undercut more likely to work.
2. **`gap_ahead`** , smaller gaps require less time to recover after the pit stop. Then there is an interesting wrinkle here too: if the gap is *too* small, the driver may not have enough room to pit cleanly.
3. **`pace_delta`** , if the pitting driver is already faster in clean air, the undercut becomes much more realistic once the fresh tires go on.
4. **`deg_delta` / `ca_deg_delta`** , the difference in degradation rates helps show which car is running into the tire drop-off first.
5. **`pit_loss`** , this is circuit-specific. Street tracks with long pit lanes, like Monaco, Singapore, and Baku, naturally make the undercut harder to pull off.

### Interpretation
There is still a real error floor here, and I think that is expected. Mechanical issues, sudden rain, radio communication problems, and driver execution all add noise that a lap-timing model cannot fully observe. So I would not expect a model like this to reach 100%, and honestly that would be suspicious. The more realistic goal is to estimate undercut probability in a useful way, not to pretend the strategy outcome is deterministic.
